<a href="https://colab.research.google.com/github/jonathancdelc/jonathancdelc/blob/main/Practica_2_Data_Science_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Preparación del entorno
!pip install python-docx --quiet

import sqlite3
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', None)
np.random.seed(42)

DB_NAME = 'dml_ml.db'
DB_PATH = os.path.join('/content', DB_NAME)
print("Ruta de la Base de Datos configurada en:", DB_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.1 MB/s eta 0:00:00
Ruta de la Base de Datos configurada en: /content/dml_ml.db


In [4]:
# Crear la base de datos de origen con datos simulados (Versión Corregida)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# 1. Crear tabla original con las 7 columnas básicas
cur.execute("""
CREATE TABLE features (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    feature1 REAL,
    feature2 REAL,
    category TEXT,
    label INTEGER,
    created_at TEXT
);
""")
conn.commit()

# 2. Generar datos simulados del archivo original
n = 30000
user_ids = np.random.randint(1000, 2000, size=n)
feature1 = np.random.exponential(scale=100.0, size=n)
outlier_idx = np.random.choice(n, size=int(n*0.005), replace=False)
feature1[outlier_idx] *= 30
feature2 = np.random.normal(loc=50, scale=15, size=n)
nan_idx = np.random.choice(n, size=int(n*0.02), replace=False)
for i in nan_idx:
    feature2[i] = np.nan

cats = ['online','store','partner','mobile']
category = np.random.choice(cats, size=n, p=[0.5,0.25,0.15,0.10])
label = np.random.choice([0,1], size=n, p=[0.85,0.15])
start = datetime.now() - timedelta(days=90)
created_at = [(start + timedelta(seconds=int(np.random.rand()*90*24*3600))).isoformat() for _ in range(n)]

# Armamos las filas con los 6 valores que pide la consulta original
rows = []
for i in range(n):
    f1 = float(feature1[i])
    if np.random.rand() < 0.005:
        f1 = None
    f2 = None if np.isnan(feature2[i]) else float(feature2[i])
    rows.append((int(user_ids[i]), f1, f2, category[i], int(label[i]), created_at[i]))

# 3. Insertar datos en lotes usando la estructura exacta de 6 campos (Evita el error de bindings)
batch_size = 5000
insert_sql = "INSERT INTO features (user_id, feature1, feature2, category, label, created_at) VALUES (?, ?, ?, ?, ?, ?);"
for i in range(0, n, batch_size):
    cur.executemany(insert_sql, rows[i:i+batch_size])
    conn.commit()

# 4. Ahora sí, añadimos las columnas extras como nulas para la práctica guiada posterior
cur.execute("ALTER TABLE features ADD COLUMN feature1_scaled REAL;")
cur.execute("ALTER TABLE features ADD COLUMN feature1_capped REAL;")
conn.commit()

print(f"¡Base de datos generada exitosamente con {n} filas iniciales y columnas extras listas!")
conn.close()

¡Base de datos generada exitosamente con 30000 filas iniciales y columnas extras listas!


In [5]:
# Conectamos a la base de datos
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Listar tablas existentes
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("--- Tablas encontradas ---")
print(tablas)

# Listar columnas de la tabla 'features'
columnas = pd.read_sql_query("PRAGMA table_info(features);", conn)
print("\n--- Columnas de la tabla 'features' ---")
print(columnas[['name', 'type']])

# Crear la copia exacta llamada features_work
cur.execute("DROP TABLE IF EXISTS features_work;")
cur.execute("CREATE TABLE features_work AS SELECT * FROM features;")
conn.commit()

# Ver conteo inicial
conteo_inicial = pd.read_sql_query("SELECT COUNT(*) as total FROM features_work;", conn)['total'][0]
print(f"\n¡Copia creada con éxito! Filas iniciales en 'features_work': {conteo_inicial}")
conn.close()

--- Tablas encontradas ---
              name
0         features
1  sqlite_sequence

--- Columnas de la tabla 'features' ---
              name     type
0               id  INTEGER
1          user_id  INTEGER
2         feature1     REAL
3         feature2     REAL
4         category     TEXT
5            label  INTEGER
6       created_at     TEXT
7  feature1_scaled     REAL
8  feature1_capped     REAL

¡Copia creada con éxito! Filas iniciales en 'features_work': 30000


Aquí lo primero que hicimos fue "asomarnos" a la base de datos para ver qué tablas teníamos y qué datos guardaba cada columna. Como no queríamos arruinar la tabla original (features) haciendo experimentos, usamos un truco de SQL para sacarle una fotocopia idéntica y guardarla en una nueva tabla llamada features_work. ¡Así podemos equivocarme y borrar cosas sin culpa!

In [6]:
# Primero preparamos la tabla de registros para cumplir con el Punto 7
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS ingest_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ingresados INTEGER,
    actualizados INTEGER,
    ignorados INTEGER,
    timestamp TEXT
);
""")
conn.commit()

# Generamos 200 registros de la categoría 'promo'
np.random.seed(123)  # para que sea reproducible
n_nuevos = 200
user_ids_nuevos = np.random.randint(1000, 2000, size=n_nuevos)
f1_nuevos = np.random.exponential(scale=100.0, size=n_nuevos)
f2_nuevos = np.random.normal(loc=50, scale=15, size=n_nuevos)
start_date = datetime.now() - timedelta(days=10)
created_at_nuevos = [(start_date + timedelta(seconds=int(np.random.rand()*10*24*3600))).isoformat() for _ in range(n_nuevos)]

# Conteo antes de insertar
antes = pd.read_sql_query("SELECT COUNT(*) as total FROM features_work;", conn)['total'][0]

# Insertamos simulando un "INSERT OR IGNORE" buscando duplicados por user_id y created_at
# Nota: Como cargamos en memoria, validamos duplicados con lógica de Python antes de meterlos
existentes = pd.read_sql_query("SELECT user_id, created_at FROM features_work;", conn)
parejas_existentes = set(zip(existentes['user_id'], existentes['created_at']))

filas_a_insertar = []
ignorados = 0

for i in range(n_nuevos):
    registro = (int(user_ids_nuevos[i]), float(f1_nuevos[i]), float(f2_nuevos[i]), 'promo', 0, created_at_nuevos[i], None, None)
    # Si la combinación user_id + fecha ya existía, la ignoramos
    if (user_ids_nuevos[i], created_at_nuevos[i]) in parejas_existentes:
        ignorados += 1
    else:
        filas_a_insertar.append(registro)

# Ejecutamos la inserción real en lote
if filas_a_insertar:
    cur.executemany("""
    INSERT INTO features_work (user_id, feature1, feature2, category, label, created_at, feature1_scaled, feature1_capped)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?);
    """, filas_a_insertar)
    conn.commit()

ingresados = len(filas_a_insertar)

# Conteo después de insertar
despues = pd.read_sql_query("SELECT COUNT(*) as total FROM features_work;", conn)['total'][0]
print(f"Filas ANTES: {antes} | Filas DESPUÉS: {despues} (Insertadas con éxito: {ingresados}, Ignoradas por duplicado: {ignorados})")

# GUARDADO DEL PUNTO 7: Registramos en la bitácora
cur.execute("INSERT INTO ingest_log (ingresados, actualizados, ignorados, timestamp) VALUES (?, ?, ?, ?);",
            (ingresados, 0, ignorados, datetime.now().isoformat()))
conn.commit()
conn.close()

Filas ANTES: 30000 | Filas DESPUÉS: 30200 (Insertadas con éxito: 200, Ignoradas por duplicado: 0)


Aquí inventamos 200 clientes nuevos que entraron por una campaña de promoción (promo). Pero en el mundo real, a veces el sistema se traba y mete dos veces el mismo cliente a la misma hora. Para evitar llenar la base de datos de basura duplicada, pusimos un filtro de seguridad: revisamos si el código del usuario y la fecha exacta ya existían en la tabla. Si ya estaban, los ignoramos, y si eran nuevos, los dejamos pasar.

In [7]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Buscamos el valor máximo de la columna feature1 en la tabla de trabajo
res = cur.execute("SELECT MAX(feature1) FROM features_work WHERE feature1 IS NOT NULL;").fetchone()
max_val = res[0] if res else 1.0
print(f"El valor máximo encontrado en feature1 es: {max_val}")

# Actualizamos dividiendo cada valor por ese máximo
cur.execute("UPDATE features_work SET feature1_scaled = feature1 / ? WHERE feature1 IS NOT NULL;", (max_val,))
conn.commit()

# Mostramos una muestra para verificar que los números ahora queden entre 0 y 1
muestra_escala = pd.read_sql_query("SELECT id, feature1, feature1_scaled FROM features_work WHERE feature1 IS NOT NULL LIMIT 5;", conn)
display(muestra_escala)
conn.close()

El valor máximo encontrado en feature1 es: 15688.594845282652


,id,feature1,feature1_scaled
0,1,7.630387,0.000486
1,2,35.346743,0.002253
2,3,11.578172,0.000738
3,4,188.657548,0.012025
4,5,97.249718,0.006199


A los modelos de inteligencia artificial no les gusta cuando una columna tiene números chiquitos (como 1 o 2) y otra tiene números gigantes (como un millón), porque se confunden y le hacen más caso al número grande. Lo que hicimos aquí fue buscar el número más grande de toda la columna (MAX) y dividir todos los demás números entre él. Así, logramos que todos los datos queden achicados a una escala limpia entre 0 y 1, haciendo que la competencia entre variables sea justa.

In [8]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Contamos antes de la purga
antes_borrar = cur.execute("SELECT COUNT(*) FROM features_work;").fetchone()[0]

# Aplicamos la regla estricta de borrado
cur.execute("""
DELETE FROM features_work
WHERE label IS NULL
   OR (feature1 IS NULL AND feature2 IS NULL);
""")
borradas = cur.rowcount
conn.commit()

print(f"¡Purga completada! Se eliminaron {borradas} filas que tenían datos rotos o nulos imposibles de salvar.")
conn.close()

¡Purga completada! Se eliminaron 3 filas que tenían datos rotos o nulos imposibles de salvar.


A veces llegan filas que están tan incompletas que es imposible usarlas para entrenar a la IA; por ejemplo, si no sabemos si el cliente compró o no (label vacío), o si no tiene ninguna de las dos características principales registradas. En lugar de intentar adivinar qué datos iban ahí, tomamos la decisión difícil de "limpiar la casa" y borrar por completo esas filas insalvables

In [9]:
conn = sqlite3.connect(DB_PATH)

# Calculamos la fecha límite de hace 60 días en formato texto
fecha_limite = (datetime.now() - timedelta(days=60)).isoformat()

# Escribimos la consulta filtrando filas limpias y fechas recientes
query_ml = f"""
SELECT id, user_id, feature1_scaled, feature2, category, label
FROM features_work
WHERE feature1_scaled IS NOT NULL
  AND feature2 IS NOT NULL
  AND created_at >= '{fecha_limite}'
"""

df_ml = pd.read_sql_query(query_ml, conn)
print(f"Dataset de Machine Learning listo. Filas obtenidas: {len(df_ml)}")

# Lo guardamos como archivo CSV
df_ml.to_csv('features_ml.csv', index=False)
print("¡Archivo 'features_ml.csv' guardado en el almacenamiento local de Colab!")
conn.close()

Dataset de Machine Learning listo. Filas obtenidas: 19706
¡Archivo 'features_ml.csv' guardado en el almacenamiento local de Colab!


Aquí es donde ocurre la magia. Filtramos la tabla para quedarnos solo con lo mejor de lo mejor: filas que no tengan vacíos y que además sean nuevitas (de los últimos 60 días), porque el comportamiento viejo ya no sirve para predecir el futuro. Al final, exportamos este resultado comprimido en un archivo portátil .csv, que es el formato estándar que cualquier científico de datos necesita para empezar a entrenar modelos.

In [10]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Consulta pesada de agregación
sql_benchmark = """
SELECT category, label, COUNT(*), AVG(feature2)
FROM features_work
GROUP BY category, label;
"""

# 1. Medir SIN índice (3 repeticiones)
tiempos_sin = []
for _ in range(3):
    t0 = time.perf_counter()
    cur.execute(sql_benchmark).fetchall()
    tiempos_sin.append(time.perf_counter() - t0)
avg_sin = np.mean(tiempos_sin)

# 2. Crear el índice acelerador
cur.execute("CREATE INDEX IF NOT EXISTS idx_features_category_label ON features_work(category, label);")
conn.commit()

# 3. Medir CON índice (3 repeticiones)
tiempos_con = []
for _ in range(3):
    t0 = time.perf_counter()
    cur.execute(sql_benchmark).fetchall()
    tiempos_con.append(time.perf_counter() - t0)
avg_con = np.mean(tiempos_con)

print("=== RESULTADOS DEL BANCO DE PRUEBAS ===")
print(f"Tiempo promedio SIN índice: {avg_sin:.6f} segundos")
print(f"Tiempo promedio CON índice: {avg_con:.6f} segundos")
print(f"¡El índice aceleró la consulta unas {avg_sin/avg_con:.1f} veces!")
conn.close()

=== RESULTADOS DEL BANCO DE PRUEBAS ===
Tiempo promedio SIN índice: 0.046072 segundos
Tiempo promedio CON índice: 0.032347 segundos
¡El índice aceleró la consulta unas 1.4 veces!


Imagina buscar una receta en un libro de 500 páginas hojeando una por una (eso es hacer una consulta sin índice). Lo que hicimos aquí fue medir cuánto tardaba la computadora en agrupar los datos a ciegas, luego le creamos un "Índice" (que funciona exactamente como el índice final de un libro) y volvimos a medir el tiempo. ¡La diferencia de velocidad es increíble porque la base de datos ahora sabe exactamente a qué página ir!

In [11]:
conn = sqlite3.connect(DB_PATH)
df_bitacora = pd.read_sql_query("SELECT * FROM ingest_log;", conn)
print("=== CONTENIDO DE LA TABLA INGEST_LOG ===")
display(df_bitacora)
conn.close()

=== CONTENIDO DE LA TABLA INGEST_LOG ===


,id,ingresados,actualizados,ignorados,timestamp
0,1,200,0,0,2026-07-18T14:47:47.733913


En los trabajos reales no puedes estar revisando las pantallas todo el tiempo para ver si los procesos automáticos funcionaron. Por eso creamos una tabla tipo "caja negra" o bitácora llamada ingest_log. Cada vez que metimos los datos de las promociones en el Punto 2, el sistema escribió un renglón en esta tabla avisándonos la hora exacta, cuántos registros metió y cuántos ignoró por seguridad.

Reflexión: El mantenimiento continuo mediante operaciones INSERT, UPDATE y DELETE actúa como el filtro de calidad definitivo en un flujo de Data Science, previniendo que registros corruptos o desactualizados sesguen las predicciones del modelo. Respecto a las variables derivadas, la regla de oro dicta que conviene persistirlas (guardarlas físicamente en columnas de la base de datos) cuando se trata de transformaciones pesadas o estáticas que serán consultadas repetidamente por múltiples procesos de entrenamiento, reduciendo así el costo computacional. Por el contrario, calcularlas "al vuelo" mediante vistas o consultas dinámicas es la opción ideal para métricas que cambian constantemente en tiempo real o dependen de ventanas de tiempo móviles, evitando el riesgo de almacenar datos obsoletos que perjudiquen el desempeño del pipeline.